#Chromadb Vector Store

In [6]:
!pip install -U langchain langchain-xai langchain-community langchain-chroma chromadb tiktoken pypdf langchain-huggingface

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

# Runs locally on your CPU/GPU
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
from langchain_chroma import Chroma

In [10]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [11]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [13]:
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory='my_chroma_db', #folder
    collection_name='sample' #in folder a collection
)

In [14]:
# add documents
vector_store.add_documents(docs)

['b6be739d-78e4-4cbb-a090-722ef16d9ffe',
 'ab06dc71-f6ab-42f3-8b40-0b45e515ee6a',
 '32b64bba-5343-4f97-8b26-30276f16a347',
 '36c10a21-72cc-4d14-964f-b58a519b2418',
 '75abdc6f-aded-4a35-b412-889e93559b7b']

In [15]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['b6be739d-78e4-4cbb-a090-722ef16d9ffe',
  'ab06dc71-f6ab-42f3-8b40-0b45e515ee6a',
  '32b64bba-5343-4f97-8b26-30276f16a347',
  '36c10a21-72cc-4d14-964f-b58a519b2418',
  '75abdc6f-aded-4a35-b412-889e93559b7b'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0.07840683, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [17]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2 #how may similar objects you want see
)

[Document(id='36c10a21-72cc-4d14-964f-b58a519b2418', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='ab06dc71-f6ab-42f3-8b40-0b45e515ee6a', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [18]:
# search with similarity score
vector_store.similarity_search_with_score( #low score better
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='36c10a21-72cc-4d14-964f-b58a519b2418', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693599343299866),
 (Document(id='ab06dc71-f6ab-42f3-8b40-0b45e515ee6a', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.149344801902771)]

In [19]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='32b64bba-5343-4f97-8b26-30276f16a347', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436005115509033),
 (Document(id='75abdc6f-aded-4a35-b412-889e93559b7b', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937328338623)]

In [20]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)

In [21]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])


{'ids': ['b6be739d-78e4-4cbb-a090-722ef16d9ffe',
  'ab06dc71-f6ab-42f3-8b40-0b45e515ee6a',
  '32b64bba-5343-4f97-8b26-30276f16a347',
  '36c10a21-72cc-4d14-964f-b58a519b2418',
  '75abdc6f-aded-4a35-b412-889e93559b7b'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0.07840683, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [22]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [23]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['b6be739d-78e4-4cbb-a090-722ef16d9ffe',
  'ab06dc71-f6ab-42f3-8b40-0b45e515ee6a',
  '32b64bba-5343-4f97-8b26-30276f16a347',
  '36c10a21-72cc-4d14-964f-b58a519b2418',
  '75abdc6f-aded-4a35-b412-889e93559b7b'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0.07840683, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca